# 02 · Silver — Validación de calidad físico-química

**Proyecto:** PoC Arquitectura Lakehouse para Monitoreo Ambiental Hídrico  
**Autora:** Kathiana Murillo García  
**Máster:** UNIR — Análisis y Visualización de Datos Masivos (2026–2027)  

## Objetivo
Aplicar 7 reglas de validación físico-química sobre los registros de la capa Bronze
y clasificar cada registro en `VALID`, `NULL_KEY` o `OUT_OF_RANGE`.

## Reglas de calidad implementadas
| Regla | Variable | Condición VALID | Referencia |
|-------|----------|-----------------|------------|
| R1 | Estación (ST.) | No nula | Protocolo NIES |
| R2 | Fecha (Date) | No nula | Protocolo NIES |
| R3 | DO0m | 0–20 mg/L | Wetzel (2001) |
| R4 | pH0m | 5–10 | Wetzel (2001) |
| R5 | WT0m | 0–40 °C | Wetzel (2001) |
| R6 | T-P | 0–500 µg/L | OECD (2008) |
| R7 | Chl-a | 0–300 µg/L | OECD (2008) |

## 1. Configuración

In [ ]:
# Celda 1 — Parámetros
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, lit, count, round as spark_round

CATALOG = "training"
SCHEMA  = "kasumigaura"
TABLE_BRONZE = f"{CATALOG}.{SCHEMA}.bronze_water_quality"
TABLE_SILVER = f"{CATALOG}.{SCHEMA}.silver_water_quality"

# Intentar detectar catálogo activo
try:
    active_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
    CATALOG = active_catalog
    TABLE_BRONZE = f"{CATALOG}.{SCHEMA}.bronze_water_quality"
    TABLE_SILVER = f"{CATALOG}.{SCHEMA}.silver_water_quality"
except Exception:
    pass

print(f"Bronze: {TABLE_BRONZE}")
print(f"Silver: {TABLE_SILVER}")

## 2. Lectura de Bronze y definición de reglas

In [ ]:
# Celda 2 — Leer capa Bronze
df_bronze = spark.table(TABLE_BRONZE)
n_bronze = df_bronze.count()
print(f"Registros Bronze: {n_bronze:,}")
df_bronze.printSchema()

In [ ]:
# Celda 3 — Aplicar 7 reglas de calidad físico-química
# Cada regla produce una columna booleana intermedia.
# La columna dq_flag se determina como:
#   VALID        : todas las reglas en True
#   NULL_KEY     : R1 (estación) o R2 (fecha) en False
#   OUT_OF_RANGE : alguna regla R3–R7 en False (valor fuera de rango)

df_silver = (
    df_bronze
    # ── R1: estación no nula ────────────────────────────────────────────────
    .withColumn("rule_ok_R1", col("ST_").isNotNull())
    # ── R2: fecha no nula ───────────────────────────────────────────────────
    .withColumn("rule_ok_R2", col("Date").isNotNull())
    # ── R3: DO0m en rango 0–20 mg/L (Wetzel 2001) ──────────────────────────
    .withColumn("rule_ok_R3",
        when(col("DO0m").isNull(), True)
        .otherwise(col("DO0m").between(0, 20)))
    # ── R4: pH0m en rango 5–10 (Wetzel 2001) ───────────────────────────────
    .withColumn("rule_ok_R4",
        when(col("pH0m").isNull(), True)
        .otherwise(col("pH0m").between(5, 10)))
    # ── R5: WT0m en rango 0–40 °C (Wetzel 2001) ────────────────────────────
    .withColumn("rule_ok_R5",
        when(col("WT0m").isNull(), True)
        .otherwise(col("WT0m").between(0, 40)))
    # ── R6: T-P en rango 0–500 µg/L (OECD 2008) ────────────────────────────
    .withColumn("rule_ok_R6",
        when(F.col("`T-P`").isNull(), True)
        .otherwise(F.col("`T-P`").between(0, 500)))
    # ── R7: Chl-a en rango 0–300 µg/L (OECD 2008) ──────────────────────────
    .withColumn("rule_ok_R7",
        when(F.col("`Chl-a`").isNull(), True)
        .otherwise(F.col("`Chl-a`").between(0, 300)))
    # ── Flag compuesto ───────────────────────────────────────────────────────
    .withColumn(
        "all_ok",
        (col("rule_ok_R1") & col("rule_ok_R2") &
         col("rule_ok_R3") & col("rule_ok_R4") &
         col("rule_ok_R5") & col("rule_ok_R6") & col("rule_ok_R7"))
    )
    .withColumn(
        "dq_flag",
        when(col("all_ok"), "VALID")
        .when(~col("rule_ok_R1") | ~col("rule_ok_R2"), "NULL_KEY")
        .when(
            ~col("rule_ok_R3") | ~col("rule_ok_R4") |
            ~col("rule_ok_R5") | ~col("rule_ok_R6") | ~col("rule_ok_R7"),
            "OUT_OF_RANGE"
        )
        .otherwise("NULL_KEY")
    )
    # Eliminar columnas auxiliares booleanas
    .drop("rule_ok_R1", "rule_ok_R2", "rule_ok_R3", "rule_ok_R4",
          "rule_ok_R5", "rule_ok_R6", "rule_ok_R7", "all_ok")
)

print("Reglas aplicadas. Vista previa del dq_flag:")
df_silver.select("ST_", "Date", "DO0m", "pH0m", "dq_flag").show(10)

## 3. Escritura en Delta Lake (capa Silver)

In [ ]:
# Celda 4 — Escribir capa Silver
(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE_SILVER)
)

n_silver = spark.table(TABLE_SILVER).count()
print(f"✓ Tabla Silver escrita: {TABLE_SILVER}")
print(f"  Registros: {n_silver:,}")